<a href="https://colab.research.google.com/github/Saad-cpp/Data-Science-Internship/blob/main/N2C2_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dir_path = "/content/drive/My Drive/Data Set/"

In [ ]:
!pip install llama-index
!pip install llama-index-embeddings-huggingface
!pip install auto-gptq
!pip install optimum
!pip install bitsandbytes
!pip install llama-index-llms-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.3/176.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.1/376.1 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing inst

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor

In [ ]:
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
Settings.llm = None
Settings.chunk_size = 256
Settings.chunk_overlap = 20

LLM is explicitly disabled. Using MockLLM.


In [ ]:
documents = SimpleDirectoryReader(dir_path).load_data()

In [ ]:
#index = VectorStoreIndex.from_documents(documents)
from llama_index.core import StorageContext, load_index_from_storage

# rebuild storage context
storage_context = StorageContext.from_defaults(persist_dir="/content/drive/MyDrive")

# load index
index = load_index_from_storage(storage_context)

In [ ]:
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=10
)

In [ ]:
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[
        SimilarityPostprocessor(similarity_cutoff=0.7)
    ]
)

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ",
    device_map="auto",
)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# model_name = "Qwen/Qwen2.5-0.5B-Instruct-GPTQ-Int8"
model_name =  "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=False, revision="main")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.16G [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:4713: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
Some weights of the model checkpoint at TheBloke/Mistral-7B-Instruct-v0.2-GPTQ were not used when initializing MistralForCausalLM: ['model.layers.0.mlp.down_proj.bias', 'model.layers.0.mlp.gate_proj.bias', 'model.layers.0.mlp.up_proj.bias', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.o_proj.bias', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.1.mlp.down_proj.bias', 'model.layers.1.mlp.gate_proj.bias', 'model.layers.1.mlp.up_proj.bias', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.o_proj.bias', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.10.mlp.down_proj.bias', 'model.layers.10.mlp.gate_proj.bias', 'model.la

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

In [ ]:
prompt_template = lambda context, question: f"""
[INST] You are a knowledgeable assistant and precise in question answering tasks. You only give answers that are derived from given information.
Your goal is to provide precise, detailed and only contextually relevant answers based on given information.
Remember: Nothing should be generated outside of context. If no context is given just say 'information not available'.
Here is context.
{context}
Given the context and no prior knowledge answer the following question. Answer should only come out of context. If you generate answer on your own, you will be
erased from the system.
# Question: {question} [/INST]"""
# prompt_template = lambda context, question: [{"role": "system", "content": f"You are a knowledgeable assistant and precise in question answering tasks. You only give answers that are given in context. Anything out of context is out of your scope. Your goal is to provide concise and only contextually relevant answers based on given information.Remember: Nothing should be generated outside of context. If no context is given just say 'information not available'. Here is context and question for you to work with. {context}"}, {"role": "user", "content": question}] #For Qwen model

In [ ]:
# question = "what illness is common in patients with Chief complaint: weakness?"
# response = query_engine.query(question)
# context = "context:\n"
# for node in response.source_nodes:
#     context += node.node.get_text()
# prompt = prompt_template(context, question)
# print(prompt)

In [ ]:
# print(len(response.source_nodes))

In [ ]:
# text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True) #For Qwen model role: user template
# inputs = tokenizer([text], return_tensors="pt").to("cuda")
# inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
# output = model.generate(inputs["input_ids"].to("cuda"), max_new_tokens = 512)
# response = tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# print(response)

In [ ]:
# print(response.split("assistant")[-1]) #For Qwen model
# print(response.split('[/INST]')[-1])

In [ ]:
import time

def rag_func(question):
  start_time = time.time()
  rep = query_engine.query(question)
  context = "context:\n"
  for node in rep.source_nodes:
      context += node.node.get_text()
  if len(rep.source_nodes) <= 0:
      context = "NO GIVEN CONTEXT"
  prompt = prompt_template(context, question)
# text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True) #For Qwen model role: user template
# inputs = tokenizer([text], return_tensors="pt").to("cuda")
  inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
  if 'pad_token_id' not in tokenizer.special_tokens_map:
      tokenizer.pad_token_id = tokenizer.eos_token_id
  output = model.generate(inputs["input_ids"].to("cuda"), max_new_tokens = 512, attention_mask=inputs["attention_mask"], pad_token_id=tokenizer.pad_token_id)
  response = tokenizer.decode(output[0], skip_special_tokens=True)
  end_time = time.time()
  return response.split('[/INST]')[-1]+"\nDocuments retrieved: "+str(len(rep.source_nodes))+"\nTime Taken: "+str(end_time-start_time)
# return response.split('assistant')[-1]+"\nDocuments retrieved: "+str(len(rep.source_nodes))+"\nTime Taken: "+str(end_time-start_time)

In [ ]:
pip install gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 10.4 MB/s eta 0:00:00


In [ ]:
import gradio as gr

In [ ]:
ui = gr.Interface(fn=rag_func, inputs="text", outputs="text", title="RAG Chatbot", description="RAG Chatbot. Retrieves information from patient notes.............")

In [ ]:
ui.launch(debug=True)

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://5a9c88d14e2525badc.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
